## BikeWale – Web Scraping and Exploratory Data Analysis
- Project domain: Two-wheeler / automobile data analytics
- Data source: BikeWale website
- Project type: Web scraping, data cleaning, feature engineering, and exploratory data analysis

## Project Overview
This project collects motorcycle information from the BikeWale website and converts the extracted information into a structured dataset for analysis. The workflow combines Python-based web scraping with data cleaning, preprocessing, feature engineering, statistical analysis, and visualization.

The analysis focuses on attributes such as motorcycle brand, listed price, mileage, engine capacity, transmission, kerb weight, fuel tank capacity, and user rating. The aim is to understand patterns within the collected sample and communicate findings that may be useful to customers, researchers, and automobile businesses.

## Business Objectives
- Understand the collected market sample: Examine the brands and motorcycle segments represented in the scraped data.
- Compare price positioning: Explore how listed prices differ across brands and price categories.
- Study performance-related attributes: Compare mileage and engine capacity across manufacturers.
- Include customer feedback: Examine the user ratings available in the collected records.
- Support informed comparisons: Present structured information that can help users compare motorcycles using multiple specifications.

## Project Objectives
1. Collect motorcycle details from BikeWale pages using Python.
2. Store the extracted records in a structured tabular dataset.
3. inspect the dataset's columns, data types, distributions, and missing values.
4. Clean text and numerical fields and handle incomplete records.
5. Create derived categories for price, engine capacity, and mileage.
6. Perform univariate, bivariate, and multivariate exploratory data analysis.
7. Visualize patterns and relationships among motorcycle attributes.
8. Summarize the findings, limitations, and possible future enhancements.

## Project Workflow
1. Select the source: Identify BikeWale pages containing motorcycle information.
2. Collect page links: Retrieve motorcycle page URLs for the selected brands.
3. Scrape details: Extract bike name, brand, price, mileage, engine capacity, transmission, weight, fuel tank capacity, rating, and fuel type where available.
4. Create the raw dataset: Combine extracted records and save them as a CSV file.
5. Inspect and clean: Review data types, missing values, duplicates, and inconsistent formats.
6. Filter and preprocess: Focus the analysis on petrol motorcycles and retain complete records for the final analysis.
7. Engineer features: Derive price, engine, and mileage categories and additional comparison measures.
8. Explore and visualize: Conduct univariate, bivariate, and multivariate analysis.
9. Interpret results: Summarize patterns, business insights, limitations, and future scope.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import os
import time
import random

from urllib.parse import urljoin, urlparse

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

sns.set_theme(style="whitegrid")

In [2]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,image/avif,"
        "image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive"
}

session = requests.Session()
session.headers.update(headers)

print("Session created successfully.")

Session created successfully.


In [3]:
brand_urls = {
    "TVS": "https://www.bikewale.com/tvs-bikes/",
    "Honda": "https://www.bikewale.com/honda-bikes/",
    "Yamaha": "https://www.bikewale.com/yamaha-bikes/",
    "Bajaj": "https://www.bikewale.com/bajaj-bikes/",
    "Hero": "https://www.bikewale.com/hero-bikes/",
    "Royal Enfield": "https://www.bikewale.com/royal-enfield-bikes/",
    "KTM": "https://www.bikewale.com/ktm-bikes/",
    "Suzuki": "https://www.bikewale.com/suzuki-bikes/",
    "Kawasaki": "https://www.bikewale.com/kawasaki-bikes/",
    "BMW": "https://www.bikewale.com/bmw-bikes/",
    "Aprilia": "https://www.bikewale.com/aprilia-bikes/",
    "Benelli": "https://www.bikewale.com/benelli-bikes/",
    "Ampere": "https://www.bikewale.com/ampere-bikes/",
    "Ather": "https://www.bikewale.com/ather-bikes/",
    "BMW": "https://www.bikewale.com/bmw-bikes/",
    "Ducati":"https://www.bikewale.com/ducati-bikes/"
}

print(len(brand_urls), "brands configured.")

15 brands configured.


In [4]:
def get_page(url, retries=3, timeout=30):

    for attempt in range(1, retries + 1):

        try:

            response = session.get(
                url,
                timeout=timeout,
                allow_redirects=True
            )

            print(
                f"Attempt {attempt} | "
                f"Status: {response.status_code} | "
                f"{url}"
            )

            if response.status_code == 200:

                return response.text

            if response.status_code in [403, 429]:

                print(
                    "Access blocked or rate limited."
                )

                return None

        except requests.exceptions.RequestException as e:

            print(
                f"Request error: {e}"
            )

        time.sleep(
            random.uniform(2, 4)
        )

    return None

In [5]:
test_url = brand_urls["TVS"]

html = get_page(test_url)

print("\nHTML received:", html is not None)

if html:
    print("HTML length:", len(html))

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    print(
        "Page title:",
        soup.title.get_text(
            strip=True
        ) if soup.title else "No title"
    )

Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/

HTML received: True
HTML length: 643461
Page title: TVS Bikes Price in India - New TVS Models 2026, Images & Specs - BikeWale


In [6]:
# Inspect the links actually returned by requests

soup = BeautifulSoup(
    html,
    "html.parser"
)

all_links = []

for a in soup.find_all(
    "a",
    href=True
):

    href = a.get("href")

    text = a.get_text(
        " ",
        strip=True
    )

    all_links.append(
        {
            "text": text,
            "href": href
        }
    )

links_df = pd.DataFrame(
    all_links
)

print(
    "Total links found:",
    len(links_df)
)

display(
    links_df.head(50)
)

Total links found: 142


,text,href
0,,/
1,TVS Raider 125,/tvs-bikes/raider-125/
2,"All-New Pulsar 125 - Loaded with TFT Intelligence Bajaj Pulsar 125 ₹ 93,491 Onwards All-New Pulsar 125 - Loaded with TFT Intelligence",/bajaj-bikes/pulsar-125/
3,TVS Ronin,/tvs-bikes/ronin/
4,TVS Apache RTR 160,/tvs-bikes/apache-160/
5,TVS Jupiter,/tvs-bikes/jupiter/
6,TVS iQube,/tvs-bikes/iqube/
7,TVS Ntorq 125,/tvs-bikes/ntorq-125/
8,TVS Jupiter 125,/tvs-bikes/jupiter-125/
9,TVS Apache RTR 160 4V,/tvs-bikes/apache-rtr-160-4v/


In [7]:
def extract_bike_urls(brand_url):

    html = get_page(brand_url)

    if not html:
        return []

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    parsed_brand_url = urlparse(
        brand_url
    )

    brand_path = (
        parsed_brand_url
        .path
        .strip("/")
    )

    # Example:
    # tvs-bikes
    # honda-bikes
    # royal-enfield-bikes

    bike_section = "/" + brand_path + "/"

    bike_urls = set()

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a.get(
            "href",
            ""
        ).strip()

        if not href:
            continue

        full_url = urljoin(
            brand_url,
            href
        )

        parsed_url = urlparse(
            full_url
        )

        # Only BikeWale
        if parsed_url.netloc not in [
            "www.bikewale.com",
            "bikewale.com"
        ]:
            continue

        path = parsed_url.path

        # Normalize
        path = "/" + path.strip("/") + "/"

        # --------------------------------------------------
        # IMPORTANT:
        # Example:
        #
        # /tvs-bikes/
        # /tvs-bikes/raider-125/
        # /tvs-bikes/ronin/
        #
        # We want the second one.
        # --------------------------------------------------

        if path.startswith(
            bike_section
        ):

            remaining = path[
                len(bike_section):
            ]

            # Remove query-like structure
            remaining = remaining.strip("/")

            # Must contain a model slug
            if remaining and "/" not in remaining:

                # Exclude known non-model pages
                excluded = [
                    "news",
                    "reviews",
                    "videos",
                    "images",
                    "compare",
                    "used",
                    "dealers",
                    "insurance",
                    "emi",
                    "accessories"
                ]

                if remaining.lower() not in excluded:

                    bike_urls.add(
                        "https://www.bikewale.com"
                        + path
                    )

    return sorted(
        bike_urls
    )

In [8]:
tvs_urls = extract_bike_urls(
    brand_urls["TVS"]
)

print(
    "TVS bike URLs found:",
    len(tvs_urls)
)

for url in tvs_urls[:30]:

    print(url)

Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/
TVS bike URLs found: 67
https://www.bikewale.com/tvs-bikes/apache-160/
https://www.bikewale.com/tvs-bikes/apache-180-2018-2019/
https://www.bikewale.com/tvs-bikes/apache-180/
https://www.bikewale.com/tvs-bikes/apache-200/
https://www.bikewale.com/tvs-bikes/apache-2006/
https://www.bikewale.com/tvs-bikes/apache-rr-310-2020-2023/
https://www.bikewale.com/tvs-bikes/apache-rr-310/
https://www.bikewale.com/tvs-bikes/apache-rtr-160-4v/
https://www.bikewale.com/tvs-bikes/apache-rtr-165rp/
https://www.bikewale.com/tvs-bikes/apache-rtr-180-2018-2019/
https://www.bikewale.com/tvs-bikes/apache-rtr-180-abs-2019/
https://www.bikewale.com/tvs-bikes/apache-rtr-310-2023-2024/
https://www.bikewale.com/tvs-bikes/apache-rtr-310/
https://www.bikewale.com/tvs-bikes/apache-rtx/
https://www.bikewale.com/tvs-bikes/apache/
https://www.bikewale.com/tvs-bikes/centra/
https://www.bikewale.com/tvs-bikes/champ/
https://www.bikewale.com/tvs-bikes/excel/
h

In [9]:
test_bike_url = tvs_urls[0]

print(
    "Testing:",
    test_bike_url
)

bike_html = get_page(
    test_bike_url
)

print(
    "HTML received:",
    bike_html is not None
)

if bike_html:

    bike_soup = BeautifulSoup(
        bike_html,
        "html.parser"
    )

    print(
        "Title:",
        bike_soup.title.get_text(
            strip=True
        )
    )

Testing: https://www.bikewale.com/tvs-bikes/apache-160/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-160/
HTML received: True
Title: TVS Apache RTR 160 Price - Mileage, Images, Colours | BikeWale


In [10]:
# Print the first 5000 characters of visible text
# only for debugging

text = bike_soup.get_text(
    " ",
    strip=True
)

print(
    text[:5000]
)

TVS Apache RTR 160 Price - Mileage, Images, Colours | BikeWale Log In AD TVS Apache RTR 160 3353 Ratings Write Review Compare Overview Price Expert Opinion Similar Street Bikes Colours Mileage Specs & Features Q&A User Reviews News Mileage - ARAI 61 kmpl Max Power 15.82 bhp Kerb Weight 137 kg View The Bike Colours Images View Variant RM Drum - Black Edition City Show price in my city ₹ 1,10,695 Check on-road price Avg. Ex-Showroom price Get October Offers Contact TVS Get in touch with TVS for best offers, test rides, EMI options, exchange benefits and much more... Get Buying Assistance Variants (8) RM Drum - Black Edition RM Drum RM Disc RM Disc Bluetooth Racing Edition Anniversary Special Edition Dual Channel ABS Dual Channel ABS with TCS Compare Compare Compare Compare Compare Compare Compare Compare TVS Apache RTR 160 Price Variants (8) Apache RTR 160 RM Drum - Black Edition Apache RTR 160 RM Drum Apache RTR 160 RM Disc Apache RTR 160 RM Disc Bluetooth Apache RTR 160 Racing Edition 

In [11]:
def extract_bike_name(soup):

    h1 = soup.find("h1")

    if h1:

        name = h1.get_text(
            " ",
            strip=True
        )

        if name:

            return re.sub(
                r"\s+",
                " ",
                name
            ).strip()

    if soup.title:

        title = soup.title.get_text(
            " ",
            strip=True
        )

        title = re.sub(
            r"\s*Price.*$",
            "",
            title,
            flags=re.IGNORECASE
        )

        return title.strip()

    return np.nan

In [12]:
def extract_price(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    patterns = [

        r"Avg\.\s*Ex-Showroom\s*price.*?₹\s*([\d,]+)",

        r"Ex-Showroom\s*price.*?₹\s*([\d,]+)",

        r"On-Road\s*Price.*?₹\s*([\d,]+)"

    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            price = match.group(1)

            price = re.sub(
                r"[^\d]",
                "",
                price
            )

            if price:

                return float(price)

    return np.nan

In [13]:
def extract_engine_capacity(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    patterns = [

        r"Engine Capacity\s*\|?\s*([\d,.]+)\s*cc",

        r"Displacement\s*\|?\s*([\d,.]+)\s*cc"

    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            value = match.group(1)

            value = value.replace(
                ",",
                ""
            )

            return float(value)

    return np.nan

In [14]:
def extract_mileage(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    patterns = [

        r"Mileage\s*-\s*ARAI\s*\|?\s*([\d,.]+)\s*kmpl",

        r"Mileage\s*-\s*ARAI\s*([\d,.]+)\s*kmpl",

        r"([\d,.]+)\s*kmpl"

    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            value = match.group(1)

            value = value.replace(
                ",",
                ""
            )

            return float(value)

    return np.nan

In [15]:
def extract_transmission(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    patterns = [

        r"Transmission\s*\|?\s*"
        r"(Automatic|Manual|"
        r"\d+\s*Speed\s*Manual|"
        r"\d+\s*Speed\s*Automatic)",

        r"Transmission\s*"
        r"(Automatic|Manual|"
        r"\d+\s*Speed\s*Manual|"
        r"\d+\s*Speed\s*Automatic)"

    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            return match.group(1).strip()

    return np.nan

In [16]:
def extract_kerb_weight(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    patterns = [

        r"Kerb Weight\s*\|?\s*([\d,.]+)\s*kg",

        r"Kerb Weight\s*([\d,.]+)\s*kg"

    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            value = match.group(1)

            value = value.replace(
                ",",
                ""
            )

            return float(value)

    return np.nan

In [17]:
def extract_fuel_tank(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    patterns = [

        r"Fuel Tank Capacity\s*\|?\s*"
        r"([\d,.]+)\s*(?:litres?|L)",

        r"Fuel Tank Capacity\s*"
        r"([\d,.]+)\s*(?:litres?|L)"

    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            value = match.group(1)

            value = value.replace(
                ",",
                ""
            )

            return float(value)

    return np.nan

In [18]:
def extract_rating(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    # Prefer User Rating if available
    user_rating_pattern = (
        r"User Rating\s*\(?\s*"
        r"([\d.]+)"
    )

    match = re.search(
        user_rating_pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:

        rating = float(
            match.group(1)
        )

        if 0 <= rating <= 5:

            return rating

    # Fallback
    ratings = re.findall(
        r"\b([0-5](?:\.\d+)?)\s*/\s*5\b",
        text
    )

    for rating in ratings:

        rating = float(rating)

        if 0 <= rating <= 5:

            return rating

    return np.nan

In [19]:
def extract_fuel_type(soup):

    text = soup.get_text(
        " ",
        strip=True
    )

    pattern = (
        r"Fuel Type\s*\|?\s*"
        r"(Petrol|Electric|CNG|Diesel|Hybrid)"
    )

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:

        return match.group(1).title()

    return np.nan

In [20]:
def scrape_bike(url, brand):

    html = get_page(url)

    if not html:

        return None

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    data = {

        "Bike_Name":
            extract_bike_name(soup),

        "Brand":
            brand,

        "Price":
            extract_price(soup),

        "Mileage":
            extract_mileage(soup),

        "Engine_Capacity":
            extract_engine_capacity(soup),

        "Transmission":
            extract_transmission(soup),

        "Kerb_Weight":
            extract_kerb_weight(soup),

        "Fuel_Tank_Capacity":
            extract_fuel_tank(soup),

        "User_Rating":
            extract_rating(soup),

        "Fuel_Type":
            extract_fuel_type(soup),

        "URL":
            url
    }

    return data

In [21]:
# Test first bike

test_result = scrape_bike(
    tvs_urls[0],
    "TVS"
)

test_result

Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-160/


{'Bike_Name': 'TVS Apache RTR 160',
 'Brand': 'TVS',
 'Price': 110695.0,
 'Mileage': 61.0,
 'Engine_Capacity': 159.7,
 'Transmission': '5 Speed Manual',
 'Kerb_Weight': 137.0,
 'Fuel_Tank_Capacity': 12.0,
 'User_Rating': 4.5,
 'Fuel_Type': 'Petrol',
 'URL': 'https://www.bikewale.com/tvs-bikes/apache-160/'}

In [22]:
# Test first bike

test_result = scrape_bike(
    tvs_urls[0],
    "TVS"
)

test_result

Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-160/


{'Bike_Name': 'TVS Apache RTR 160',
 'Brand': 'TVS',
 'Price': 110695.0,
 'Mileage': 61.0,
 'Engine_Capacity': 159.7,
 'Transmission': '5 Speed Manual',
 'Kerb_Weight': 137.0,
 'Fuel_Tank_Capacity': 12.0,
 'User_Rating': 4.5,
 'Fuel_Type': 'Petrol',
 'URL': 'https://www.bikewale.com/tvs-bikes/apache-160/'}

In [23]:
# ============================================================
# COLLECT BIKE URLs FOR ALL BRANDS
# ============================================================

all_bike_urls = []

for brand, brand_url in brand_urls.items():

    print("\n" + "=" * 60)

    print(
        "Scraping brand:",
        brand
    )

    print("=" * 60)

    try:

        urls = extract_bike_urls(
            brand_url
        )

        print(
            "URLs found:",
            len(urls)
        )

        for url in urls:

            all_bike_urls.append(
                {
                    "Brand": brand,
                    "URL": url
                }
            )

        time.sleep(
            random.uniform(2, 4)
        )

    except Exception as e:

        print(
            "Error:",
            e
        )


bike_urls_df = pd.DataFrame(
    all_bike_urls
)

bike_urls_df = bike_urls_df.drop_duplicates(
    subset=["URL"]
)

print(
    "\nTotal unique URLs:",
    len(bike_urls_df)
)


Scraping brand: TVS
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/
URLs found: 67

Scraping brand: Honda
Attempt 1 | Status: 200 | https://www.bikewale.com/honda-bikes/
URLs found: 140

Scraping brand: Yamaha
Attempt 1 | Status: 200 | https://www.bikewale.com/yamaha-bikes/
URLs found: 107

Scraping brand: Bajaj
Attempt 1 | Status: 200 | https://www.bikewale.com/bajaj-bikes/
URLs found: 96

Scraping brand: Hero
Attempt 1 | Status: 200 | https://www.bikewale.com/hero-bikes/
URLs found: 87

Scraping brand: Royal Enfield
Attempt 1 | Status: 404 | https://www.bikewale.com/royal-enfield-bikes/
Attempt 2 | Status: 404 | https://www.bikewale.com/royal-enfield-bikes/
Attempt 3 | Status: 404 | https://www.bikewale.com/royal-enfield-bikes/
URLs found: 0

Scraping brand: KTM
Attempt 1 | Status: 200 | https://www.bikewale.com/ktm-bikes/
URLs found: 57

Scraping brand: Suzuki
Attempt 1 | Status: 200 | https://www.bikewale.com/suzuki-bikes/
URLs found: 74

Scraping brand: Kawasaki
Att

In [24]:
# Check how many URLs were found for every brand

brand_url_counts = (
    bike_urls_df
    .groupby("Brand")
    .size()
    .sort_values(
        ascending=False
    )
)

display(
    brand_url_counts
)

Brand
Kawasaki    144
Honda       140
Ducati      110
Yamaha      107
Bajaj        96
Hero         87
Suzuki       74
BMW          70
TVS          67
KTM          57
Aprilia      37
Benelli      23
Ampere       14
Ather         8
dtype: int64

In [25]:
# ============================================================
# SCRAPE ALL BIKE DETAILS
# ============================================================

scraped_data = []

failed_urls = []

total = len(
    bike_urls_df
)

for index, row in bike_urls_df.iterrows():

    brand = row["Brand"]

    url = row["URL"]

    print(
        f"[{index + 1}/{total}] "
        f"{brand} | {url}"
    )

    try:

        result = scrape_bike(
            url,
            brand
        )

        if result:

            scraped_data.append(
                result
            )

        else:

            failed_urls.append(
                url
            )

    except Exception as e:

        print(
            "ERROR:",
            e
        )

        failed_urls.append(
            url
        )

    # IMPORTANT:
    # Keep a delay between requests.
    time.sleep(
        random.uniform(2, 4)
    )


print("\nScraping finished.")

print(
    "Successful:",
    len(scraped_data)
)

print(
    "Failed:",
    len(failed_urls)
)

[1/1034] TVS | https://www.bikewale.com/tvs-bikes/apache-160/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-160/
[2/1034] TVS | https://www.bikewale.com/tvs-bikes/apache-180-2018-2019/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-180-2018-2019/
[3/1034] TVS | https://www.bikewale.com/tvs-bikes/apache-180/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-180/
[4/1034] TVS | https://www.bikewale.com/tvs-bikes/apache-200/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-200/
[5/1034] TVS | https://www.bikewale.com/tvs-bikes/apache-2006/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-2006/
[6/1034] TVS | https://www.bikewale.com/tvs-bikes/apache-rr-310-2020-2023/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-rr-310-2020-2023/
[7/1034] TVS | https://www.bikewale.com/tvs-bikes/apache-rr-310/
Attempt 1 | Status: 200 | https://www.bikewale.com/tvs-bikes/apache-rr-310/
[

In [26]:
raw_df = pd.DataFrame(
    scraped_data
)

raw_df = raw_df.drop_duplicates(
    subset=[
        "Brand",
        "Bike_Name"
    ]
)

print(
    "Raw dataset shape:",
    raw_df.shape
)

display(
    raw_df.head(20)
)

Raw dataset shape: (1032, 11)


,Bike_Name,Brand,Price,Mileage,Engine_Capacity,Transmission,Kerb_Weight,Fuel_Tank_Capacity,User_Rating,Fuel_Type,URL
0,TVS Apache RTR 160,TVS,110695.0,61.0,159.70,5 Speed Manual,137.0,12.0,4.5,Petrol,https://www.bikewale.com/tvs-bikes/apache-160/
1,TVS Apache RTR 180 [2018-2019],TVS,110695.0,46.0,177.00,5 Speed Manual,139.0,16.0,4.6,Petrol,https://www.bikewale.com/tvs-bikes/apache-180-2018-2019/
2,TVS Apache RTR 180,TVS,127722.0,42.0,177.40,5 Speed Manual,140.0,12.0,4.6,Petrol,https://www.bikewale.com/tvs-bikes/apache-180/
3,TVS Apache RTR 200 4V,TVS,138353.0,41.9,197.75,5 Speed Manual,152.0,12.0,4.5,Petrol,https://www.bikewale.com/tvs-bikes/apache-200/
4,TVS Apache [2006],TVS,110695.0,18.0,NaN,NaN,NaN,NaN,4.5,NaN,https://www.bikewale.com/tvs-bikes/apache-2006/
5,TVS Apache RR 310 [2020-2023],TVS,110695.0,30.0,312.20,6 Speed Manual,169.0,11.0,4.7,Petrol,https://www.bikewale.com/tvs-bikes/apache-rr-310-2020-2023/
6,TVS Apache RR 310,TVS,262268.0,34.7,312.20,6 Speed Manual,174.0,11.0,1.0,Petrol,https://www.bikewale.com/tvs-bikes/apache-rr-310/
7,TVS Apache RTR 160 4V,TVS,116479.0,41.4,159.70,5 Speed Manual,144.0,12.0,4.5,Petrol,https://www.bikewale.com/tvs-bikes/apache-rtr-160-4v/
8,TVS Apache RTR 165RP,TVS,110695.0,45.0,165.00,5 Speed Manual,148.0,12.0,4.4,Petrol,https://www.bikewale.com/tvs-bikes/apache-rtr-165rp/
9,TVS Apache RTR 180 [2019],TVS,110695.0,42.0,177.00,5 Speed Manual,139.0,16.0,4.8,Petrol,https://www.bikewale.com/tvs-bikes/apache-rtr-180-2018-2019/


In [27]:
raw_df.to_csv(
    "raw_bike_data.csv",
    index=False
)

print(
    "Raw dataset saved successfully."
)

Raw dataset saved successfully.


In [28]:
 # KEEP PETROL/CONVENTIONAL BIKES FOR THIS PROJECT
 

raw_df["Fuel_Type"] = (
    raw_df["Fuel_Type"]
    .astype("string")
    .str.strip()
)

petrol_df = raw_df[
    raw_df["Fuel_Type"].eq("Petrol")
].copy()

print(
    "Total scraped records:",
    len(raw_df)
)

print(
    "Petrol bike records:",
    len(petrol_df)
)

print(
    "Excluded non-petrol records:",
    len(raw_df) - len(petrol_df)
)

Total scraped records: 1032
Petrol bike records: 802
Excluded non-petrol records: 230


In [29]:
df = petrol_df.copy()

In [30]:
raw_df.to_csv("bikewale_raw_data.csv", index=False)

In [31]:
 print("Dataset Shape:", df.shape)

Dataset Shape: (802, 11)


In [32]:
print("\nColumn Names:")
print(df.columns.tolist())


Column Names:
['Bike_Name', 'Brand', 'Price', 'Mileage', 'Engine_Capacity', 'Transmission', 'Kerb_Weight', 'Fuel_Tank_Capacity', 'User_Rating', 'Fuel_Type', 'URL']


In [33]:
print("\nMissing Values:")
print(df.isnull().sum())



Missing Values:
Bike_Name               0
Brand                   0
Price                   0
Mileage                16
Engine_Capacity         2
Transmission            5
Kerb_Weight            45
Fuel_Tank_Capacity     33
User_Rating           181
Fuel_Type               0
URL                     0
dtype: int64


In [34]:
print("\nDuplicate Rows:", df.duplicated().sum())


Duplicate Rows: 0


In [35]:
print("\nData Types:")
print(df.dtypes)


Data Types:
Bike_Name                     object
Brand                         object
Price                        float64
Mileage                      float64
Engine_Capacity              float64
Transmission                  object
Kerb_Weight                  float64
Fuel_Tank_Capacity           float64
User_Rating                  float64
Fuel_Type             string[python]
URL                           object
dtype: object
